In [1]:
%reload_ext autoreload
%autoreload 2

In [2]:
alphapeptdeep_hdf = r'D:\optimising-plasma-proteomics\alphapept\alphapeptdeep\output\output.speclib.hdf'
top_k_frag = 16

In [3]:
frag_inten = 0.001


min_frag_mz = 200
max_frag_mz = 1800
min_frag_nAA = 0

output_diann_tsv = (
    f"{alphapeptdeep_hdf[:-len('.speclib.hdf')]}_frags={top_k_frag}.speclib.tsv"
)
output_diann_tsv

'D:\\optimising-plasma-proteomics\\alphapept\\alphapeptdeep\\output\\output_frags=16.speclib.tsv'

In [4]:
from peptdeep.protein.fasta import PredictSpecLibFasta

fasta_lib = PredictSpecLibFasta(
    None,
    decoy=None
)

d:\optimising-plasma-proteomics\alphapept\alphapeptdeep\peptdeep-venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
d:\optimising-plasma-proteomics\alphapept\alphapeptdeep\peptdeep-venv\Lib\site-packages\peptdeep\model\ms2.py:416: UserWarning: mask_modloss is deprecated and will be removed in the future. To mask the modloss fragments, the charged_frag_types should not include the modloss fragments.
  warnings.warn(


In [5]:
fasta_lib.load_hdf(alphapeptdeep_hdf, load_mod_seq=True)

In [6]:
if 'id' in fasta_lib.protein_df.columns:
    fasta_lib.protein_df.rename(columns={'id':'protein_id'}, inplace=True)

In [7]:
import os, psutil
import numpy as np
process = psutil.Process(os.getpid())
print(f'{len(fasta_lib.precursor_df)*1e-6:.2f}M precursors with {np.prod(fasta_lib.fragment_mz_df.values.shape, dtype=float)*(1e-6):.2f}M fragments used {process.memory_info().rss/1024**3:.4f} GB memory')

9.00M precursors with 630.34M fragments used 2.5096 GB memory


In [8]:
fasta_lib.append_protein_name()

In [9]:
import pyarrow as pa
import pyarrow.parquet as pq
parquet_file = pq.ParquetFile(r"D:\DIANN-beg\outputs\trial-lib-to-parquet\report-lib.parquet")
df = parquet_file.read(columns=['Precursor.Id', 'Decoy']).to_pandas().drop_duplicates(subset=['Precursor.Id'])
len(df)
parquet_file.schema_arrow

Precursor.Id: string not null
Modified.Sequence: string not null
Stripped.Sequence: string not null
Precursor.Charge: int64 not null
Proteotypic: int64 not null
Decoy: int64 not null
N.Term: int64 not null
C.Term: int64 not null
RT: float not null
IM: float not null
Q.Value: float not null
Peptidoform.Q.Value: float not null
PTM.Site.Confidence: float not null
PG.Q.Value: float not null
Precursor.Mz: float not null
Product.Mz: float not null
Relative.Intensity: float not null
Fragment.Type: string not null
Fragment.Charge: int64 not null
Fragment.Series.Number: int64 not null
Fragment.Loss.Type: string not null
Exclude.From.Quant: int64 not null
Protein.Ids: string not null
Protein.Group: string not null
Protein.Names: string not null
Genes: string not null
Flags: int64 not null